# Week 1 Deep Dive: Transformers & Tokens, Explained Simply

This notebook is a hands-on companion to **Section 1.1 ("The Transformer Demystified")** of the Week 1 chapter. It has two goals:

1. **Tokens** — see exactly how text gets chopped into the numbered "Lego bricks" a model actually reads.
2. **Attention** — build a tiny, from-scratch attention mechanism with NumPy so you can watch a word "decide" which other words matter to it.

No API key or GPU needed — everything here runs locally with `tiktoken` and `numpy`.

## 1. Setup and Installation

In [ ]:
!pip install tiktoken numpy matplotlib --quiet

In [ ]:
import tiktoken
import numpy as np
import matplotlib.pyplot as plt

## 2. Tokens: Lego Bricks for Language

Models never see letters or words — only **token IDs**, numbers from a fixed vocabulary. `tiktoken` is OpenAI's tokenizer library; we'll use the `cl100k_base` encoding (used by GPT-4) since it matches the examples from the chapter.

In [ ]:
encoding = tiktoken.get_encoding("cl100k_base")

examples = ["hello", "ChatGPT", "unbelievable", "2024-01-15"]

for text in examples:
    token_ids = encoding.encode(text)
    pieces = [encoding.decode([tid]) for tid in token_ids]
    print(f"{text!r:16} -> {len(token_ids)} token(s): {pieces} -> ids {token_ids}")

Notice how `"hello"` stays whole (it's common), while `"ChatGPT"` and `"unbelievable"` get split into smaller, more frequent sub-pieces. This is **Byte Pair Encoding (BPE)**: the tokenizer merges the most frequent adjacent character pairs until it reaches a fixed vocabulary size (`cl100k_base` has ~100k tokens).

Try your own text below — change `my_text` and re-run the cell.

In [ ]:
my_text = "Tokenization is the first step of every transformer."

token_ids = encoding.encode(my_text)
pieces = [encoding.decode([tid]) for tid in token_ids]

print(f"Text: {my_text!r}")
print(f"Token count: {len(token_ids)}")
print(f"Pieces: {pieces}")
print(f"Approx. words: {len(my_text.split())} (rule of thumb: 1 token \u2248 0.75 words)")

## 3. Attention: Every Word Asks a Question

Recall the sentence from the chapter: *"The animal didn't cross the street because it was too tired."* A human resolves "it" → "animal" instantly. Let's build a tiny, from-scratch version of **scaled dot-product attention** and watch the model do the same thing.

We'll assign each word a small hand-crafted embedding vector where semantically related words (like "it" and "animal") point in similar directions — exactly what a trained embedding table would learn on its own after seeing enough text.

In [ ]:
words = ["The", "animal", "didn't", "cross", "the", "street", "because", "it", "was", "tired"]

# Hand-crafted 4-dim embeddings: 'animal' and 'it' share a strong signal in the first dimension
# to stand in for what a trained embedding table would discover from real data.
np.random.seed(42)
embeddings = {
    "The": [0.1, 0.2, 0.0, 0.1],
    "animal": [0.9, 0.1, 0.2, 0.0],
    "didn't": [0.0, 0.6, 0.1, 0.2],
    "cross": [0.1, 0.5, 0.6, 0.1],
    "the": [0.1, 0.2, 0.0, 0.1],
    "street": [0.2, 0.1, 0.8, 0.1],
    "because": [0.0, 0.3, 0.1, 0.5],
    "it": [0.85, 0.15, 0.25, 0.05],
    "was": [0.1, 0.4, 0.0, 0.3],
    "tired": [0.7, 0.2, 0.1, 0.6],
}

X = np.array([embeddings[w] for w in words])
print("Embedding matrix shape:", X.shape)

In [ ]:
def scaled_dot_product_attention(Q, K, V):
    """Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V"""
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    # softmax over each row
    exp_scores = np.exp(scores - scores.max(axis=-1, keepdims=True))
    weights = exp_scores / exp_scores.sum(axis=-1, keepdims=True)
    output = weights @ V
    return output, weights

# For self-attention, Query, Key, and Value all come from the same input (X).
# Real transformers first project X through learned W_Q, W_K, W_V matrices; we skip
# that here to keep the demo focused on the attention mechanism itself.
output, attn_weights = scaled_dot_product_attention(X, X, X)

it_index = words.index("it")
print(f"Attention weights for the word 'it':\n")
for word, weight in sorted(zip(words, attn_weights[it_index]), key=lambda p: -p[1]):
    print(f"  {word:10} {weight:.3f}")

Look at the top of that list — "it" pays the most attention to "animal" (and to itself), exactly like the intuition from the chapter. In a real trained model, these embeddings and the attention weighting are *learned* automatically from billions of examples; here we hand-crafted the embeddings just to make the mechanism visible.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(attn_weights, cmap="viridis")
ax.set_xticks(range(len(words)))
ax.set_yticks(range(len(words)))
ax.set_xticklabels(words, rotation=45, ha="right")
ax.set_yticklabels(words)
ax.set_xlabel("Attending TO")
ax.set_ylabel("Attending FROM")
ax.set_title("Self-Attention Weights")
fig.colorbar(im, ax=ax, label="Attention weight")
plt.tight_layout()
plt.show()

## 4. Multi-Head Attention in One Cell

A real transformer doesn't run attention once — it runs it many times in parallel ("heads"), each with its own learned Q/K/V projections, so different heads can specialize in different kinds of relationships (grammar, meaning, position, etc.). Here we simulate two heads with two different random projections of the same embeddings.

In [ ]:
def random_projection(X, out_dim, seed):
    rng = np.random.default_rng(seed)
    W = rng.normal(scale=0.5, size=(X.shape[1], out_dim))
    return X @ W

num_heads = 2
head_dim = 4

for head in range(num_heads):
    Q = random_projection(X, head_dim, seed=head * 10 + 1)
    K = random_projection(X, head_dim, seed=head * 10 + 2)
    V = random_projection(X, head_dim, seed=head * 10 + 3)
    _, weights = scaled_dot_product_attention(Q, K, V)
    top_word = words[np.argmax(weights[it_index])]
    print(f"Head {head}: 'it' attends most to -> {top_word!r}")

Different heads can (and often do) focus on different relationships — this is exactly why stacking many heads lets a transformer capture grammar, meaning, and position simultaneously.

## 5. Checkpoint

Before moving on, make sure you can answer these:

1. Why does `"ChatGPT"` tokenize into 3 tokens instead of 1, while `"hello"` tokenizes into just 1?
2. In the attention formula `softmax(QK^T / sqrt(d_k))V`, what would happen to the output if every attention weight were equal (uniform)?
3. Why do transformers use *multiple* attention heads instead of just one?

> 💡 **Further reading:** See the "Further Reading" section of the Week 1 chapter for the Nebius tokenizer guide and the original *Attention Is All You Need* paper.